# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A7mad7-7/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Content Refresh Prioritization & Reason Code Assignment

We convert raw model prediction probabilities into a prioritized content refresh queue by combining the predicted probability of decay (`decay_probability`) with the historical search demand volume (`impressions_90d`).

**Prioritization Formula:**
$$\text{Priority Score} = \text{Decay Probability} \times \log(1 + \text{Impressions}_{90d})$$

**Reason Code Mapping:**
1. **`HIGH_DEMAND_STALE`:** High traffic volume ($\ge 75\text{th}$ percentile) with extreme staleness ($>180$ days).
2. **`CTR_DECAY_RISK`:** Predicted decay probability $> 0.60$ with falling CTR performance.
3. **`LEGACY_STAGNANT`:** Stale content ($>365$ days) with moderate demand.
4. **`LOW_PRIORITY_MONITOR`:** Low impression volume or low decay probability.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Ensure output directories exist
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

# 1. Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Filter active valid content
valid_mask = (df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)
df_clean = df[valid_mask].copy()

# Feature Engineering
df_clean['ctr'] = df_clean['clicks_90d'] / df_clean['impressions_90d']
df_clean['log_impressions'] = np.log1p(df_clean['impressions_90d'])
df_clean['log_clicks'] = np.log1p(df_clean['clicks_90d'])
df_clean['target_decay'] = (df_clean['trend_direction'] == 'down').astype(int)

feature_cols = ['impressions_90d', 'clicks_90d', 'days_since_last_update', 'content_age_days', 'ctr', 'log_impressions', 'log_clicks']
X = df_clean[feature_cols]
y = df_clean['target_decay']
groups = df_clean['client_id']

# 2. Fit Final Validated Random Forest Model
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced')
rf_model.fit(X.iloc[train_idx], y.iloc[train_idx])

# Generate probabilities across all candidate pages
df_clean['decay_prob'] = rf_model.predict_proba(X)[:, 1]

# Compute Priority Score
df_clean['priority_score'] = df_clean['decay_prob'] * df_clean['log_impressions']

# Assign Reason Codes
high_imp_cutoff = df_clean['impressions_90d'].quantile(0.75)

conditions = [
    (df_clean['impressions_90d'] >= high_imp_cutoff) & (df_clean['days_since_last_update'] > 180),
    (df_clean['decay_prob'] >= 0.60) & (df_clean['ctr'] < df_clean['ctr'].median()),
    (df_clean['days_since_last_update'] > 365)
]
choices = ['HIGH_DEMAND_STALE', 'CTR_DECAY_RISK', 'LEGACY_STAGNANT']

df_clean['reason_code'] = np.select(conditions, choices, default='LOW_PRIORITY_MONITOR')

# Sort Queue by Priority Score
ranked_queue = df_clean.sort_values(by='priority_score', ascending=False).reset_index(drop=True)

print("--- Ranked Queue Sample (Top 5 Priority Actions) ---")
print(ranked_queue[['content_id', 'client_id', 'priority_score', 'decay_prob', 'reason_code', 'impressions_90d']].head())

--- Ranked Queue Sample (Top 5 Priority Actions) ---
             content_id          client_id  priority_score  decay_prob  \
0  content_a023517539fe  client_6208ef0f77        8.160624    0.664873   
1  content_54baba704595  client_6208ef0f77        7.885239    0.669373   
2  content_453722754fea  client_f369cb89fc        7.857423    0.663075   
3  content_39881853ef0c  client_f369cb89fc        7.781504    0.669081   
4  content_a7c2dfc8a6ec  client_7f2253d7e2        7.759510    0.689743   

      reason_code  impressions_90d  
0  CTR_DECAY_RISK           214047  
1  CTR_DECAY_RISK           130617  
2  CTR_DECAY_RISK           140079  
3  CTR_DECAY_RISK           112434  
4  CTR_DECAY_RISK            76868  


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use Operational Boundary & Technical Limits

**Intended Users & Workflow Integration:**
* **Primary Users:** Content Strategists, SEO Managers, and Editorial Leads.
* **Operational Mode:** Semi-automated decision-support batch tool evaluated monthly or quarterly.
* **Scope:** Ranks existing published content candidates for manual refresh intervention.

**Explicit Operational Limits:**
1. **No Automatic Execution:** The model provides decision-support rankings; it does not rewrite, delete, or modify published articles automatically.
2. **Static Feature Horizon:** Predictions reflect 90-day historical aggregates and cannot anticipate sudden external shifts (e.g., Google Core Algorithm updates or viral trend shifts) occurring in real-time.
3. **Domain Authority Agnostic:** The model evaluates page-level features and does not account for site-wide technical SEO issues or domain penalty changes.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compute Queue Summary Statistics by Reason Code
queue_summary = ranked_queue.groupby('reason_code').agg(
    total_pages=('content_id', 'count'),
    mean_decay_prob=('decay_prob', 'mean'),
    mean_impressions=('impressions_90d', 'mean'),
    mean_staleness_days=('days_since_last_update', 'mean')
).reset_index().sort_values(by='total_pages', ascending=False)

print("--- Queue Summary Distribution by Reason Code ---")
print(queue_summary.to_string(index=False))

--- Queue Summary Distribution by Reason Code ---
         reason_code  total_pages  mean_decay_prob  mean_impressions  mean_staleness_days
LOW_PRIORITY_MONITOR        23998         0.458089       5819.695308            44.065339
      CTR_DECAY_RISK         5988         0.678413       2698.862892            53.751169
   HIGH_DEMAND_STALE            9         0.573009      21012.111111           193.777778
     LEGACY_STAGNANT            5         0.360398          8.200000           372.600000


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human-in-the-Loop Protocol & The No-Go List

To prevent operational harm, all high-priority refresh recommendations must pass a mandatory human editorial verification gate.

#### Human Review Checklist:
- [ ] **Search Intent Verification:** Confirm user search intent for the target query has not fundamentally shifted.
- [ ] **Fact & Link Audit:** Verify technical accuracy, statistics, and broken outbound links.
- [ ] **Brand Alignment:** Ensure tone and product positioning conform to current brand guidelines.

#### The Mandatory No-Go List (STRICTLY PROHIBITED FROM AUTOMATION):
1. **Legal & Compliance Pages:** Terms of Service, Privacy Policy, Medical/Financial Disclaimer pages.
2. **Core Conversion / Product Landing Pages:** Pricing pages and main product conversion funnels.
3. **Automated Content Deletion:** Content removal or URL redirects must never be automated without human approval.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simulate No-Go Safeguard Filter Logic
# (E.g., Flagging system pages or restricted client/content categories)

ranked_queue['review_status'] = 'PENDING_HUMAN_REVIEW'

# Apply No-Go Policy Safeguard
# Example Rule: Legacy pages with zero traffic in 90d require full structural review before refresh
no_go_condition = (ranked_queue['impressions_90d'] == 0) | (ranked_queue['reason_code'] == 'LOW_PRIORITY_MONITOR')
ranked_queue.loc[no_go_condition, 'review_status'] = 'EXCLUDED_OR_NO_ACTION'

status_counts = ranked_queue['review_status'].value_counts()

print("--- Human Review Status Safeguard Counts ---")
print(status_counts)

--- Human Review Status Safeguard Counts ---
review_status
EXCLUDED_OR_NO_ACTION    23998
PENDING_HUMAN_REVIEW      6002
Name: count, dtype: int64


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Model Monitoring & Retrain Triggers

To prevent model degradation and stale recommendations over time, the following operational retrain triggers are established:

1. **Performance Degradation Trigger:** If human-verified refresh precision drops below **50%** over two consecutive review cycles.
2. **Data Drift Trigger:** If the mean dataset staleness (`days_since_last_update`) or CTR distribution shifts by more than **20%** compared to baseline training data.
3. **Search Engine Algorithm Shift:** Immediate model retraining triggered following major search engine core updates.
4. **Time-based Cadence:** Automatic scheduled quarterly retrain cycle using updated 90-day performance windows.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Monitoring Check Function
def check_model_health_triggers(current_df, baseline_mean_imp, baseline_mean_stale):
    curr_mean_imp = current_df['impressions_90d'].mean()
    curr_mean_stale = current_df['days_since_last_update'].mean()
    
    imp_drift = abs(curr_mean_imp - baseline_mean_imp) / baseline_mean_imp
    stale_drift = abs(curr_mean_stale - baseline_mean_stale) / baseline_mean_stale
    
    print(f"Impression Drift: {imp_drift:.2%}")
    print(f"Staleness Drift: {stale_drift:.2%}")
    
    if imp_drift > 0.20 or stale_drift > 0.20:
        return "TRIGGER_RETRAIN: Distribution drift exceeds 20% threshold!"
    return "HEALTHY: Model remains within operating parameters."

health_status = check_model_health_triggers(
    ranked_queue, 
    baseline_mean_imp=df_clean['impressions_90d'].mean(), 
    baseline_mean_stale=df_clean['days_since_last_update'].mean()
)

print(f"\nModel Health Check Result: {health_status}")

Impression Drift: 0.00%
Staleness Drift: 0.00%

Model Health Check Result: HEALTHY: Model remains within operating parameters.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Artifact Exports for Research Paper & Downstream Execution

We export the finalized priority action queue CSV and key evaluation metrics JSON to `work/outputs/` and `work/figures/` for inclusion in the final research paper report.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import matplotlib.pyplot as plt

# 1. Export Ranked Priority Queue CSV (Target directory: work/outputs/)
export_cols = ['content_id', 'client_id', 'priority_score', 'decay_prob', 'reason_code', 'review_status', 'impressions_90d', 'days_since_last_update']
export_queue_path = "../outputs/priority_action_queue.csv"
ranked_queue[export_cols].to_csv(export_queue_path, index=False)
print(f"Exported priority queue ({len(ranked_queue):,} rows) to: {export_queue_path}")

# 2. Export Metrics JSON Receipts (Target directory: work/outputs/)
metrics_payload = {
    "total_candidates_analyzed": int(len(ranked_queue)),
    "pending_human_review_count": int((ranked_queue['review_status'] == 'PENDING_HUMAN_REVIEW').sum()),
    "high_demand_stale_count": int((ranked_queue['reason_code'] == 'HIGH_DEMAND_STALE').sum()),
    "ctr_decay_risk_count": int((ranked_queue['reason_code'] == 'CTR_DECAY_RISK').sum()),
    "legacy_stagnant_count": int((ranked_queue['reason_code'] == 'LEGACY_STAGNANT').sum()),
    "mean_decay_probability": float(ranked_queue['decay_prob'].mean())
}

metrics_json_path = "../outputs/playbook_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(metrics_payload, f, indent=4)
print(f"Exported metrics payload receipt to: {metrics_json_path}")

# 3. Export Figure Visualization (Target directory: work/figures/)
plt.figure(figsize=(8, 5))
ranked_queue['reason_code'].value_counts().plot(kind='bar', color='#1f77b4')
plt.title("Content Refresh Queue Distribution by Reason Code")
plt.xlabel("Reason Code")
plt.ylabel("Number of Content Pages")
plt.xticks(rotation=15)
plt.tight_layout()

figure_path = "../figures/action_distribution.png"
plt.savefig(figure_path, dpi=300)
plt.close()
print(f"Exported queue distribution figure to: {figure_path}")

Exported priority queue (30,000 rows) to: ../outputs/priority_action_queue.csv
Exported metrics payload receipt to: ../outputs/playbook_metrics.json
Exported queue distribution figure to: ../figures/action_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.